# GMFlow 玩具模型训练示例

此 Notebook 将 `train_toymodel.py` 脚本改写为易于阅读和运行的 Notebook，并添加中文注释。

In [ ]:
import os
import sys
sys.path.append(os.path.abspath('.'))  # 将当前目录加入搜索路径
import tqdm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import torch

from lib.models import GMFlowMLP2DDenoiser, GMFlow
from lib.datasets import CheckerboardData

# 允许在 GPU 上使用 TF32 加速
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

D = 2  # 数据维度
EPS = 1e-4  # 避免除零的数值稳定项
T_SCALE = 1000  # 时间步缩放
N_TEST_SAMPLES = int(1e6)  # 测试时要生成的样本数量

# 超参数，可根据需要调整
num_gaussians = 32  # 高斯混合的成分数
batch_size = 4096  # 训练批大小
num_iters = 50000  # 训练迭代次数
lr = 2e-4  # 学习率
num_steps = 8  # 采样时的步数
out_path = 'train_toymodel.png'  # 结果图像保存路径

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def gm_kl_loss(gm, sample, eps=1e-4):
    """高斯混合模型的 KL 散度损失（省略常数项）

    参数:
        gm: 包含 mean/std/weight 的字典
        sample: (bs, D) 的真实梯度 u
    返回:
        (bs,) 的损失向量
    """
    means = gm['means']
    logstds = gm['logstds']
    logweights = gm['logweights']

    inverse_stds = torch.exp(-logstds).clamp(max=1 / eps)
    diff_weighted = (sample.unsqueeze(-2) - means) * inverse_stds  # (bs, num_gaussians, D)
    gaussian_ll = (-0.5 * diff_weighted.square() - logstds).sum(dim=-1)  # (bs, num_gaussians)
    gm_nll = -torch.logsumexp(gaussian_ll + logweights.squeeze(-1), dim=-1)  # (bs,)
    return gm_nll

In [ ]:
# 构建去噪器与数据集
denoiser = GMFlowMLP2DDenoiser(num_gaussians=num_gaussians).to(device)
dataset = CheckerboardData(scale=4)
sample_batches = dataset.samples.to(device).split(batch_size, dim=0)
optimizer = torch.optim.Adam(denoiser.parameters(), lr=lr)

In [ ]:
# 训练循环
for i in range(num_iters):
    x_0 = sample_batches[i % len(sample_batches)]
    optimizer.zero_grad()

    t = torch.rand(x_0.size(0), device=device).clamp(min=EPS)  # 随机时间步
    noise = torch.randn_like(x_0)  # 生成标准高斯噪声

    sigma = t
    alpha = 1 - sigma
    x_t = alpha.unsqueeze(-1) * x_0 + sigma.unsqueeze(-1) * noise  # 添加噪声
    u = noise - x_0  # (x_t - x_0) / sigma

    u_gm = denoiser(x_t, t * T_SCALE)  # 网络预测的 u
    loss = gm_kl_loss(u_gm, u)

    loss.mean().backward()
    optimizer.step()

    if i % 1000 == 0:
        print(f'Iter {i}, loss: {loss.mean().item()}')

print('Training finished. Starting inference...')

In [ ]:
# 推断并生成样本直方图
torch.set_grad_enabled(False)

model = GMFlow(
    denoising=denoiser,
    num_timesteps=T_SCALE,
    test_cfg=dict(  # 使用二阶 GM-SDE 求解器
        output_mode='sample',
        sampler='GMFlowSDE',
        num_timesteps=num_steps,
        order=2)
).eval()

samples = []
for _ in tqdm.tqdm(range(int(N_TEST_SAMPLES // batch_size))):
    noise = torch.randn((batch_size, D, 1, 1), device=device)
    samples.append(model.forward_test(noise=noise).reshape(batch_size, D).cpu().numpy())
samples = np.concatenate(samples, axis=0)

histo, _, _ = np.histogram2d(
    samples[:, 0], samples[:, 1], bins=200, range=[[-4.2, 4.2], [-4.2, 4.2]])
histo_image = (histo.T[::-1] / 160).clip(0, 1)
histo_image = cm.viridis(histo_image)
histo_image = np.round(histo_image * 255).clip(min=0, max=255).astype(np.uint8)

out_path = os.path.abspath(out_path)
out_dir = os.path.dirname(out_path)
os.makedirs(out_dir, exist_ok=True)
plt.imsave(out_path, histo_image)

print(f'Sample histogram saved to {out_path}.')